# Eye Tracking Data Analysis Validation

This notebook demonstrates the workflow for automating MATLAB processing and analyzing eye movement data in Python.

In [2]:
import os
import pandas as pd
from process_eye_data import process_subjects
from analyze_movements import process_fixations_to_movements, calculate_indices

# Paths
PROJECT_ROOT = r"c:\Projects\Thesis_EyeTracking"
SOURCE_DIR = os.path.join(PROJECT_ROOT, "Eyedata")
OUTPUT_DIR = PROJECT_ROOT # CSVs will be in ./function/
CSV_DIR = os.path.join(OUTPUT_DIR, "function")

## 1. Automated Processing (MATLAB Wrapper)
We call the MATLAB function `Convert_eye_data` for each subject found in `processed`.

In [3]:
# Run the processing (Comment out if already run or if MATLAB is not available)
# process_subjects(SOURCE_DIR, OUTPUT_DIR, PROJECT_ROOT)

# Check generated files
if os.path.exists(CSV_DIR):
    print("Generated CSVs:", os.listdir(CSV_DIR))
else:
    print(f"Directory {CSV_DIR} does not exist yet.")

Generated CSVs: ['551.csv', '889.csv', '992.csv', 'results_889.mat']


## 2. Load and Analyze Data
We load a generated CSV and apply the scanpath classification logic (Horizontal vs Vertical).

In [ ]:
# Pick a subject to analyze
SUBJECT_ID = "992"
csv_path = os.path.join(CSV_DIR, f"{SUBJECT_ID}.csv")

if os.path.exists(csv_path):
    # Define Dummy Trial/Block Order for demonstration
    # In a real scenario, this should match the experiment structure.
    # Assuming e.g. 10 trials per block, 2 blocks.
    num_rows = len(pd.read_csv(csv_path, header=None))
    print(f"Found {num_rows} trials in CSV.")
    
    # Generate dummy metadata
    # Example: trials 1-5 Block 1, 6-10 Block 2 (Adjust as needed)
    trial_order = list(range(1, num_rows + 1))
    # Split evenly into 2 blocks for demo
    cutoff = num_rows // 2
    block_order = [1] * cutoff + [2] * (num_rows - cutoff)
    
    # 1. Generate Long Format Table
    long_table = process_fixations_to_movements(SUBJECT_ID, csv_path, trial_order, block_order)
    
    print("\nLong Format Table (First 10 rows):")
    display(long_table.head(10))
    
    # 2. Calculate Indices (Validate against Dummy output)
    indices = calculate_indices(long_table)
    
    print("\nCalculated Indices per Trial:")
    display(indices.head())
    
    # Overall Block Stats
    block_stats = indices.groupby('block')['index'].mean()
    print("\nAverage Index per Block:")
    print(block_stats)
    
else:
    print(f"CSV for Subject {SUBJECT_ID} not found. Please run step 1.")

Found 16 trials in CSV.

Long Format Table (First 10 rows):


,participant_id,movement_type,trial,block
0,992,other,1,1
1,992,v,1,1
2,992,v,1,1
3,992,other,1,1
4,992,v,1,1
5,992,other,1,1
6,992,v,1,1
7,992,other,2,1
8,992,v,2,1
9,992,h,2,1



Calculated Indices per Trial:


movement_type,participant_id,block,trial,h,v,total,index
0,992,1,1,0,4,4,0.000000
1,992,1,2,1,2,3,0.333333
2,992,1,3,0,3,3,0.000000
3,992,1,4,1,0,1,1.000000
4,992,1,5,1,3,4,0.250000



Average Index per Block:
block
1    0.240774
2    0.312500
Name: index, dtype: float64
